In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
df=pd.read_csv('../data/raw/housing.csv')

In [3]:
# Split Data into Input and Output features
X=df.drop('median_house_value', axis=1)
y=df['median_house_value']

In [4]:
X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2,random_state=42)

In [5]:
X_train.shape, X_test.shape

((16512, 9), (4128, 9))

In [6]:
num_features=X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features=X_train.select_dtypes(include=['object']).columns.tolist()

# numerical feature preprocessing steps
num_transformer=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

# categorical features preprocessing steps
cat_transformer=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# Preprocess pipeline
preprocess= ColumnTransformer(
    transformers=[
        ('num',num_transformer,num_features),
        ('cat',cat_transformer,cat_features)
    ]
)



C:\Users\prashant\AppData\Local\Temp\ipykernel_7872\2303007469.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features=X_train.select_dtypes(include=['object']).columns.tolist()


In [7]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
import joblib


# Baseline Model (no cv, no Tunning)

In [8]:
baseline_pipe=Pipeline(
    steps=[("preprocess",preprocess),
           ('model',LinearRegression())]

)

In [9]:
# Preprocess the data and train the baseline model
baseline_pipe.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['longitude','latitude','housing_median_age',...,'households', 'median_income','ocean_proximity']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. 

In [10]:
train_baseline_predict=baseline_pipe.predict(X_train)
test_baseline_predict=baseline_pipe.predict(X_test)

In [11]:
train_baseline_rmse=root_mean_squared_error(y_train,train_baseline_predict)
train_baseline_mae=mean_absolute_error(y_train,train_baseline_predict)
train_baseline_r2=r2_score(y_train,train_baseline_predict)

print(f"RMSE : {train_baseline_rmse:.3f}")
print(f"MAE : {train_baseline_mae:.3f}")
print(f"R2 : {train_baseline_r2:.3f}")

RMSE : 68433.937
MAE : 49594.842
R2 : 0.650


In [12]:
test_baseline_rmse=root_mean_squared_error(y_test,test_baseline_predict)
test_baseline_mae=mean_absolute_error(y_test,test_baseline_predict)
test_baseline_r2=r2_score(y_test,test_baseline_predict)

print(f"RMSE : {test_baseline_rmse:.3f}")
print(f"MAE : {test_baseline_mae:.3f}")
print(f"R2 : {test_baseline_r2:.3f}")

RMSE : 70059.193
MAE : 50670.489
R2 : 0.625


In [13]:
# both train and test results are nearlly same and show descent perfromance, it can be case of underfitting

In [14]:
# Model Selection and Optimization
models={
    "LinearRegression":LinearRegression(),
    "Ridge":Ridge(random_state=42),
    "Lasso":Lasso(random_state=42, max_iter=10000),
    'RandomForest':RandomForestRegressor(),
    'HistGB':HistGradientBoostingRegressor()
    }

In [15]:
k=5
cv=KFold(n_splits=k,shuffle=True,random_state=42)

In [16]:
scoring={
    "rmse":"neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2":"r2"
}

In [17]:
rows=[]
for name, model in models.items():
    pipe=Pipeline(
        steps=[
            ("preprocess",preprocess),
            ("model", model)
        ]
    )
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    rows.append({
        "model": name,
        "cv_rmse": -scores["test_rmse"].mean(),
        "cv_mae": -scores['test_mae'].mean(),
        "cv_r2": scores["test_r2"].mean()
    })
cv_results=pd.DataFrame(rows).sort_values('cv_rmse')
print("=== CV model Comparison ===")
print(cv_results)

=== CV model Comparison ===
              model       cv_rmse        cv_mae     cv_r2
4            HistGB  48178.803737  32235.978039  0.826310
3      RandomForest  49437.048689  32288.027664  0.817134
1             Ridge  68595.617399  49664.330927  0.647760
2             Lasso  68603.233277  49667.262611  0.647685
0  LinearRegression  68604.162955  49667.159067  0.647676


In [18]:
best_model=cv_results.iloc[0]
best_model

model            HistGB
cv_rmse    48178.803737
cv_mae     32235.978039
cv_r2           0.82631
Name: 4, dtype: object

#### BEST MODEL : HistGradientBoostingRegressor

In [19]:
hgb_pipe=Pipeline(
    steps=[
        ('preprocess',preprocess),
        ('model', HistGradientBoostingRegressor(random_state=42))
    ]
)

In [20]:
# Hyperparameters combination
param_grid = {
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [None, 3, 6],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [20, 50, 100],
    "model__l2_regularization": [0.0, 0.1, 1.0]
}

In [21]:
grid = GridSearchCV(
    estimator=hgb_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

In [22]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__l2_regularization': [0.0, 0.1, ...], 'model__learning_rate': [0.03, 0.05, ...], 'model__max_depth': [None, 3, ...], 'model__max_leaf_nodes': [15, 31, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.S

In [23]:
print("Tunned histGB(CV)")
print("Best CV rmse : ", -grid.best_score_)
print("Best Params : ", grid.best_params_)


Tunned histGB(CV)
Best CV rmse :  47327.96653299736
Best Params :  {'model__l2_regularization': 0.0, 'model__learning_rate': 0.1, 'model__max_depth': None, 'model__max_leaf_nodes': 63, 'model__min_samples_leaf': 20}


In [24]:
# Retrain 
hgb_best = grid.best_estimator_

In [25]:
# Training predictions
train_final_pred = hgb_best.predict(X_train)

# Training metrics
train_final_rmse = root_mean_squared_error(y_train, train_final_pred)
train_final_mae = mean_absolute_error(y_train, train_final_pred)
train_final_r2 = r2_score(y_train, train_final_pred)

print("\n=== FINAL MODEL (Tuned HGB) Train Performance ===")
print(f"RMSE : {train_final_rmse:.3f}")
print(f"MAE  : {train_final_mae:.3f}")
print(f"R2   : {train_final_r2:.3f}")


=== FINAL MODEL (Tuned HGB) Train Performance ===
RMSE : 36062.562
MAE  : 24559.875
R2   : 0.903


In [26]:
# Test predictions
test_final_pred = hgb_best.predict(X_test)

# Test metrics
test_final_rmse = root_mean_squared_error(y_test, test_final_pred)
test_final_mae = mean_absolute_error(y_test, test_final_pred)
test_final_r2 = r2_score(y_test, test_final_pred)

print("\n=== FINAL MODEL (Tuned HGB) Test Performance ===")
print(f"RMSE : {test_final_rmse:.3f}")
print(f"MAE  : {test_final_mae:.3f}")
print(f"R2   : {test_final_r2:.3f}")


=== FINAL MODEL (Tuned HGB) Test Performance ===
RMSE : 46923.352
MAE  : 30990.885
R2   : 0.832


In [27]:
joblib.dump(hgb_best, "../models/final_hgb_model.pkl")

['../models/final_hgb_model.pkl']

In [28]:
print("Final model is HistGradientBoostingRegressor after hyperparameter tuning.")
print("It performs better than baseline Linear Regression based on RMSE, MAE and R2 score.")

Final model is HistGradientBoostingRegressor after hyperparameter tuning.
It performs better than baseline Linear Regression based on RMSE, MAE and R2 score.
